# PharmaCorp Analysis
Four business questions answered directly from the extracted patient records.

In [2]:
import json, statistics
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

records = json.loads(Path('../data/results_postprocessed.json').read_text())
N = len(records)

TIERS = {'high': {'high'}, 'high+med': {'high', 'medium'}, 'all': {'high', 'medium', 'low'}}

print(f'Loaded {N} records')

Loaded 50 records


## Q1 — What percentage of patients appear to be on a biologic?

In [3]:
def tier_pivot(value_fn, certainty_fn):
    '''Build a status × certainty-tier table with a Total row.'''
    rows = {}
    for tier, allowed in TIERS.items():
        counts = Counter(value_fn(r) for r in records if certainty_fn(r) in allowed)
        rows[tier] = counts
    df = pd.DataFrame(rows).fillna(0).astype(int)
    df.loc['Total'] = df.sum()
    return df.apply(lambda col: col.apply(lambda n: f'{int(n)} ({int(n)/N:.0%})')).rename_axis('status')

print(f'Total patients: {N}\n')

print('Call 1 — on_biologic (quick detection)')
display(tier_pivot(
    lambda r: r['call1']['biologic_status']['on_biologic'],
    lambda r: r['call1']['biologic_status']['certainty'],
))

print('\nCall 3 — pathway_endpoint (full journey classification)')
display(tier_pivot(
    lambda r: r['call3']['pathway_endpoint']['status'],
    lambda r: r['call3']['pathway_endpoint']['certainty'],
))

c1_yes = sum(1 for r in records if r['call1']['biologic_status']['on_biologic'] == 'yes')
c3_bio = sum(1 for r in records if r['call3']['pathway_endpoint']['status'] == 'on_biologic')
both   = sum(1 for r in records if r['call1']['biologic_status']['on_biologic'] == 'yes'
                                and r['call3']['pathway_endpoint']['status'] == 'on_biologic')
print(f'\nCross-validation — call1=yes: {c1_yes}  |  call3=on_biologic: {c3_bio}  |  both agree: {both}')

Total patients: 50

Call 1 — on_biologic (quick detection)


,high,high+med,all
status,,,
yes,37 (74%),37 (74%),37 (74%)
no,13 (26%),13 (26%),13 (26%)
Total,50 (100%),50 (100%),50 (100%)



Call 3 — pathway_endpoint (full journey classification)


,high,high+med,all
status,,,
on_biologic,37 (74%),37 (74%),37 (74%)
churned,2 (4%),2 (4%),2 (4%)
explicitly_rejected,3 (6%),3 (6%),3 (6%)
discussed_no_decision,8 (16%),8 (16%),8 (16%)
Total,50 (100%),50 (100%),50 (100%)



Cross-validation — call1=yes: 37  |  call3=on_biologic: 37  |  both agree: 37


## Q2 — For patients not on a biologic, what are the primary reasons?

In [4]:
no_bio       = [r for r in records if r['call1']['biologic_status']['on_biologic'] == 'no' and r.get('call2')]
with_reasons = [r for r in no_bio if r['call2'].get('reasons')]
n_no, n_wr   = len(no_bio), len(with_reasons)

print(f'Non-biologic patients (call1=no):  {n_no}')
print(f'  With reasons:     {n_wr}  ← denominator for % below')
print(f'  Without reasons:  {n_no - n_wr}  (biologic not yet reached or mentioned)\n')

status_counts = Counter(r['call2']['biologic_status_detail']['status'] for r in no_bio)
display(pd.DataFrame([
    {'biologic_status_detail': s, 'count': n, '%': f'{n/n_no:.0%}'}
    for s, n in status_counts.most_common()
]).set_index('biologic_status_detail'))

Non-biologic patients (call1=no):  13
  With reasons:     12  ← denominator for % below
  Without reasons:  1  (biologic not yet reached or mentioned)



,count,%
biologic_status_detail,,
discussed_no_decision,9,69%
explicitly_rejected,3,23%
not_mentioned,1,8%


In [5]:
# Barrier categories — patient-level counts across all 3 certainty tiers
tier_data = {}
for tier, allowed in TIERS.items():
    cat_patients = defaultdict(set)
    cat_mentions = Counter()
    for i, r in enumerate(with_reasons):
        for reason in r['call2']['reasons']:
            if reason['certainty'] in allowed:
                cat_patients[reason['category']].add(i)
                cat_mentions[reason['category']] += 1
    tier_data[tier] = (cat_patients, cat_mentions)

all_cats = sorted(tier_data['all'][0], key=lambda c: -len(tier_data['all'][0][c]))
rows = []
for cat in all_cats:
    row = {'barrier': cat}
    for tier in TIERS:
        n = len(tier_data[tier][0].get(cat, set()))
        row[f'{tier}  patients'] = f'{n} ({n/n_wr:.0%})'
        row[f'{tier}  mentions'] = tier_data[tier][1].get(cat, 0)
    rows.append(row)

# Total row: unique patients with any barrier (union, not sum) + total mentions
total_row = {'barrier': 'Total (any barrier)'}
for tier in TIERS:
    all_pts = set().union(*tier_data[tier][0].values()) if tier_data[tier][0] else set()
    n = len(all_pts)
    total_row[f'{tier}  patients'] = f'{n} ({n/n_wr:.0%})'
    total_row[f'{tier}  mentions'] = sum(tier_data[tier][1].values())
rows.append(total_row)

pd.DataFrame(rows).set_index('barrier')

,high patients,high mentions,high+med patients,high+med mentions,all patients,all mentions
barrier,,,,,,
cost_insurance,7 (58%),7,8 (67%),8,8 (67%),8
patient_fear,4 (33%),6,4 (33%),6,4 (33%),6
doctor_choice,3 (25%),3,3 (25%),3,3 (25%),3
access,1 (8%),1,1 (8%),1,1 (8%),1
other,0 (0%),0,1 (8%),1,1 (8%),1
Total (any barrier),11 (92%),17,12 (100%),19,12 (100%),19


## Q3 — What other treatments are commonly tried before a biologic?

In [6]:
# On-biologic patients — pre-biologic escalation ladder (biologics excluded to avoid circular counting)
bio = [r for r in records if r['call1']['biologic_status']['on_biologic'] == 'yes']

class_patients = defaultdict(set)
drug_patients  = defaultdict(set)
drug_statuses  = defaultdict(Counter)

for i, r in enumerate(bio):
    for t in r['call1']['treatment_journey']:
        dc   = t.get('drug_class') or 'Unknown'
        name = t.get('canonical_name') or t['name']
        if dc == 'Biologic':
            continue
        class_patients[dc].add(i)
        drug_patients[name].add(i)
        drug_statuses[name][t['status']] += 1

print(f'On-biologic patients ({len(bio)}) — pre-biologic treatment ladder\n')

# Drug class table with Total row (unique patients across all classes)
all_pts_class = set().union(*class_patients.values())
display(pd.DataFrame([
    {'drug_class': dc, 'patients': len(pts), '% of all 50': f'{len(pts)/N:.0%}'}
    for dc, pts in sorted(class_patients.items(), key=lambda x: -len(x[1]))
] + [{'drug_class': 'Total', 'patients': len(all_pts_class), '% of all 50': f'{len(all_pts_class)/N:.0%}'}]
).set_index('drug_class'))

# Top drugs table with Total row (unique patients across top 15; sums for status columns)
top_drugs = sorted(drug_patients.items(), key=lambda x: -len(x[1]))[:15]
all_pts_top = set().union(*[drug_patients[d] for d, _ in top_drugs])
display(pd.DataFrame([
    {'drug': d, 'patients': len(drug_patients[d]), '% of all 50': f'{len(drug_patients[d])/N:.0%}',
     'stopped': drug_statuses[d].get('stopped', 0), 'currently_on': drug_statuses[d].get('currently_on', 0),
     'discussed': drug_statuses[d].get('discussed', 0), 'rejected': drug_statuses[d].get('rejected', 0)}
    for d, _ in top_drugs
] + [{'drug': 'Total (top 15)', 'patients': len(all_pts_top), '% of all 50': f'{len(all_pts_top)/N:.0%}',
      'stopped':      sum(drug_statuses[d].get('stopped', 0)      for d, _ in top_drugs),
      'currently_on': sum(drug_statuses[d].get('currently_on', 0) for d, _ in top_drugs),
      'discussed':    sum(drug_statuses[d].get('discussed', 0)    for d, _ in top_drugs),
      'rejected':     sum(drug_statuses[d].get('rejected', 0)     for d, _ in top_drugs)}]
).set_index('drug'))

On-biologic patients (37) — pre-biologic treatment ladder



,patients,% of all 50
drug_class,,
Immunomodulator,31,62%
Aminosalicylate (5-ASA),28,56%
Corticosteroid,26,52%
Adjunct / symptom management,4,8%
Other,3,6%
Supplement / probiotic,1,2%
Total,36,72%


,patients,% of all 50,stopped,currently_on,discussed,rejected
drug,,,,,,
mesalamine (5-ASA),27,54%,27,0,0,0
methotrexate (MTX),25,50%,15,6,5,0
prednisone,24,48%,22,2,0,0
azathioprine (AZA),17,34%,15,2,0,0
sulfasalazine,5,10%,5,0,0,0
mercaptopurine (6-MP),2,4%,2,0,0,0
budesonide,2,4%,1,1,0,0
NSAIDs (unspecified),2,4%,2,0,0,0
hydroxychloroquine,1,2%,0,1,0,0


In [7]:
# Non-biologic patients — treatments tried or currently on
non_bio = [r for r in records if r['call1']['biologic_status']['on_biologic'] != 'yes']

class_patients2 = defaultdict(set)
drug_patients2  = defaultdict(set)
drug_statuses2  = defaultdict(Counter)

for i, r in enumerate(non_bio):
    for t in r['call1']['treatment_journey']:
        dc   = t.get('drug_class') or 'Unknown'
        name = t.get('canonical_name') or t['name']
        class_patients2[dc].add(i)
        drug_patients2[name].add(i)
        drug_statuses2[name][t['status']] += 1

print(f'Non-biologic patients ({len(non_bio)}) — current and tried treatments\n')

# Drug class table with Total row
all_pts_class2 = set().union(*class_patients2.values())
display(pd.DataFrame([
    {'drug_class': dc, 'patients': len(pts), '% of all 50': f'{len(pts)/N:.0%}'}
    for dc, pts in sorted(class_patients2.items(), key=lambda x: -len(x[1]))
] + [{'drug_class': 'Total', 'patients': len(all_pts_class2), '% of all 50': f'{len(all_pts_class2)/N:.0%}'}]
).set_index('drug_class'))

# Top drugs table with Total row
top_drugs2 = sorted(drug_patients2.items(), key=lambda x: -len(x[1]))[:15]
all_pts_top2 = set().union(*[drug_patients2[d] for d, _ in top_drugs2])
display(pd.DataFrame([
    {'drug': d, 'patients': len(drug_patients2[d]), '% of all 50': f'{len(drug_patients2[d])/N:.0%}',
     'stopped': drug_statuses2[d].get('stopped', 0), 'currently_on': drug_statuses2[d].get('currently_on', 0),
     'discussed': drug_statuses2[d].get('discussed', 0), 'rejected': drug_statuses2[d].get('rejected', 0)}
    for d, _ in top_drugs2
] + [{'drug': 'Total', 'patients': len(all_pts_top2), '% of all 50': f'{len(all_pts_top2)/N:.0%}',
      'stopped':      sum(drug_statuses2[d].get('stopped', 0)      for d, _ in top_drugs2),
      'currently_on': sum(drug_statuses2[d].get('currently_on', 0) for d, _ in top_drugs2),
      'discussed':    sum(drug_statuses2[d].get('discussed', 0)    for d, _ in top_drugs2),
      'rejected':     sum(drug_statuses2[d].get('rejected', 0)     for d, _ in top_drugs2)}]
).set_index('drug'))

Non-biologic patients (13) — current and tried treatments



,patients,% of all 50
drug_class,,
Aminosalicylate (5-ASA),13,26%
Corticosteroid,11,22%
Biologic,10,20%
Immunomodulator,5,10%
Supplement / probiotic,2,4%
Adjunct / symptom management,1,2%
Antibiotic,1,2%
Total,13,26%


,patients,% of all 50,stopped,currently_on,discussed,rejected
drug,,,,,,
mesalamine (5-ASA),13,26%,10,3,0,0
prednisone,10,20%,10,0,0,0
biologic (unspecified),8,16%,0,0,8,0
budesonide,5,10%,1,4,0,0
methotrexate (MTX),3,6%,1,2,0,0
sulfasalazine,3,6%,1,2,0,0
azathioprine (AZA),2,4%,0,2,0,0
ustekinumab (Stelara),1,2%,0,0,1,0
iron supplementation,1,2%,0,1,0,0


## Q4 — What does a typical referral pathway look like?

In [8]:
BIOLOGIC_PRESCRIBERS = {'Gastroenterologist', 'Rheumatologist'}

relevant = [r for r in records
            if r['call3']['pathway_endpoint']['status'] in {'on_biologic', 'discussed_no_decision', 'explicitly_rejected'}]

lengths = [len(r['call3']['referral_pathway']) for r in relevant]
print(f'Patients (on_biologic + discussed_no_decision + explicitly_rejected): {len(relevant)}\n')
print(f'Steps to biologic prescriber — mean: {statistics.mean(lengths):.1f}  median: {int(statistics.median(lengths))}  range: {min(lengths)}–{max(lengths)}')
print(f'Distribution: {dict(sorted(Counter(lengths).items()))}')

no_specialist = [r for r in relevant
                 if not r['call3']['referral_pathway']
                 or r['call3']['referral_pathway'][-1].get('canonical_provider') not in BIOLOGIC_PRESCRIBERS]
if no_specialist:
    last_steps = Counter(r['call3']['referral_pathway'][-1].get('canonical_provider', 'empty') for r in no_specialist)
    print(f'\nNote: {len(no_specialist)} patient(s) pathway ends without a named specialist ({dict(last_steps)}).')
    print('Golden-set evaluation suggests some of these are implied GI visits the model did not extract')
    print('(e.g. "Referred immediately. Colonoscopy at 19 was terrifying." without naming the specialty).')
    print('True median is likely still 2 steps; these cases bias the mean slightly downward.')

Patients (on_biologic + discussed_no_decision + explicitly_rejected): 48

Steps to biologic prescriber — mean: 1.7  median: 2  range: 1–3
Distribution: {1: 21, 2: 22, 3: 5}

Note: 13 patient(s) pathway ends without a named specialist ({'GP': 10, 'Emergency': 2, 'Health Center': 1}).
Golden-set evaluation suggests some of these are implied GI visits the model did not extract
(e.g. "Referred immediately. Colonoscopy at 19 was terrifying." without naming the specialty).
True median is likely still 2 steps; these cases bias the mean slightly downward.


In [9]:
# Transition frequencies between consecutive providers
transitions = Counter()
for r in relevant:
    path = r['call3']['referral_pathway']
    for i in range(len(path) - 1):
        a = path[i].get('canonical_provider')   or path[i]['name']
        b = path[i+1].get('canonical_provider') or path[i+1]['name']
        transitions[(a, b)] += 1

display(pd.DataFrame([
    {'transition': f'{a} → {b}', 'count': n, '%': f'{n/len(relevant):.0%}'}
    for (a, b), n in transitions.most_common(15)
]).set_index('transition'))

# Full pathway sequences
sequences = Counter(
    ' → '.join(step.get('canonical_provider') or step['name'] for step in r['call3']['referral_pathway'])
    for r in relevant
)
print()
display(pd.DataFrame([
    {'sequence': seq, 'count': n} for seq, n in sequences.most_common(10)
]).set_index('sequence'))

,count,%
transition,,
GP → Gastroenterologist,16,33%
Emergency → Gastroenterologist,5,10%
Gastroenterologist → Gastroenterologist,2,4%
GP → GP,2,4%
Health Center → Emergency,1,2%
Health Center → GP,1,2%
Health center → GP,1,2%
Mental Health Provider → Gastroenterologist,1,2%
Gynecologist → GP,1,2%


,count
sequence,
GP → Gastroenterologist,13
GP,9
Gastroenterologist,9
Emergency → Gastroenterologist,5
Health Center → Emergency,1
Health Center → GP → Gastroenterologist,1
Gastroenterologist → Gastroenterologist → Gastroenterologist,1
Health center → GP → Gastroenterologist,1
Mental Health Provider → Gastroenterologist,1


### Referral pathway — Sankey diagram

In [12]:
nodes, seen = [], set()
for (a, b) in transitions:
    for name in (a, b):
        if name not in seen:
            nodes.append(name)
            seen.add(name)
idx = {name: i for i, name in enumerate(nodes)}

fig = go.Figure(go.Sankey(
    node=dict(label=nodes, pad=15, thickness=20),
    link=dict(
        source=[idx[a] for (a, b) in transitions],
        target=[idx[b] for (a, b) in transitions],
        value=[transitions[(a, b)] for (a, b) in transitions],
    ),
))
fig.update_layout(title_text='Referral pathways — biologic-relevant patients', height=600, width=800, font_size=12)
fig.show()
